**环境说明** ℹ:
- 推荐先在 `~/.local/share/jupyter/kernels/bagel/kernel.json` 的 `env` 字段预设
  `CUDA_VISIBLE_DEVICES=0,1` (本仓库已配), 然后启动 jupyter / 在 VSCode 选 kernel.
- cell 2 也会 `setdefault` 兜底 — 若您是在 code-server 浏览器 VSCode 里运行, 且
  kernel 启动时 env 没注入, 重启 kernel 一次即可 (Ctrl+Shift+P → Jupyter: Restart Kernel).
- torch 在首次 import 时锁定可见设备, 后续 `os.environ` 修改不生效 (见 `DEBUG_NOTES.md §8.4`).

## 1. 环境 / import

In [1]:
import os
import sys
import json
import time
import random
import gc
from copy import deepcopy
from contextlib import contextmanager

# ── CUDA_VISIBLE_DEVICES ──
# 必须在 import torch 之前设置 (torch 在首次 import 时锁定可见设备,
# 见 DEBUG_NOTES.md §8.4). 优先用 kernel.json env (code-server 启动时自动注入);
# 若缺失则 setdefault 兜底 — 但如果前序 cell 已 import 过 torch, 必须 Restart Kernel.
_pre_cvd = os.environ.get("CUDA_VISIBLE_DEVICES")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")
if _pre_cvd is None:
    print(f"⚠ CUDA_VISIBLE_DEVICES 未在 kernel 启动时注入, 已用 setdefault 设为 {os.environ['CUDA_VISIBLE_DEVICES']}")
    print(f"  这是本 notebook 第一个 import torch 的 cell — 兜底生效, 继续即可.")
else:
    print(f"CUDA_VISIBLE_DEVICES = {_pre_cvd} (从 kernel.json env 注入)")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

_proj_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _proj_root not in sys.path:
    sys.path.insert(0, _proj_root)
os.chdir(_proj_root)  # 后续相对路径都以 repo root 为基准
print(f"project root: {_proj_root}")

import numpy as np
import torch
import pandas as pd
print(f"torch {torch.__version__}, visible GPUs: {torch.cuda.device_count()}")
if torch.cuda.device_count() < 2:
    print(f"⚠ 仅 {torch.cuda.device_count()} 个 GPU 可见 — asym 模式需要 2 张.")
    print(f"  如 kernel.json env 没生效, 请 Ctrl+Shift+P → 'Jupyter: Restart Kernel' 强制重启.")
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GiB)")

⚠ CUDA_VISIBLE_DEVICES 未在 kernel 启动时注入, 已用 setdefault 设为 0,1
  这是本 notebook 第一个 import torch 的 cell — 兜底生效, 继续即可.
project root: /home/wuwenxuan03/bagel
torch 2.5.1+cu121, visible GPUs: 2
  cuda:0 = NVIDIA GeForce RTX 4090 (23.6 GiB)
  cuda:1 = NVIDIA GeForce RTX 4090 (23.6 GiB)


## 2. 加载模型 (asym 放置)

调用 `load_model_asym` 完整复现 `run_cap_sweep_mp.py:230-271` 单卡 CPU 加载路径, 
然后按 `_moe_gen` 规则切分到两张卡。

In [2]:
from experiments.asym_placement import load_model_asym, verify_placement

MODEL_PATH = os.path.join(_proj_root, "BAGEL-7B-MoT")
UND_DEVICE = "cuda:0"
GEN_DEVICE = "cuda:1"

print(f"[load] loading model from {MODEL_PATH} ...")
t0 = time.perf_counter()
model, vae_model, tokenizer, new_token_ids = load_model_asym(
    model_path=MODEL_PATH,
    und_device=UND_DEVICE,
    gen_device=GEN_DEVICE,
)
print(f"[load] done in {time.perf_counter() - t0:.1f}s")

[load] loading model from /home/wuwenxuan03/bagel/BAGEL-7B-MoT ...
[asym] building CPU model from /home/wuwenxuan03/bagel/BAGEL-7B-MoT ...
[asym] vit_model → cuda:0
[asym] aux modules ['time_embedder', 'vae2llm', 'llm2vae', 'latent_pos_embed', 'vit_pos_embed', 'connector'] → cuda:0
[asym] language_model partitioned: _moe_gen → cuda:1, rest → cuda:0
[asym] vae_model → cuda:0
[load] done in 239.6s


## 3. 验证 placement

遍历 `named_parameters + named_buffers`, 断言所有 `_moe_gen` 在 gen_device, 其余在 und_device。
VAE 全部在 und_device。

In [3]:
stats = verify_placement(
    model, und_device=UND_DEVICE, gen_device=GEN_DEVICE, vae_model=vae_model, strict=True,
)
print()
print("Per-device summary:")
for dev in (UND_DEVICE, GEN_DEVICE):
    p = stats["params"][dev]
    b = stats["bytes"][dev]
    print(f"  {dev}: {p/1e9:.3f}B params, {b/1024**3:.2f} GiB")

[verify] und_device (cuda:0): 8.165B params, 15.37 GiB
[verify] gen_device (cuda:1): 6.526B params, 12.15 GiB
[verify] ✅ all params/buffers on expected devices

Per-device summary:
  cuda:0: 8.165B params, 15.37 GiB
  cuda:1: 6.526B params, 12.15 GiB


## 4. P0 修复 — 装 input transfer shim

`bagel.py` 的所有 `prepare_*` 方法返回的 `generation_input` 全是 CPU tensor。
pipeline 模式靠 `accelerate force_hooks=True` 自动搬运, asym 无 hook 必须显式补这层。
装 6 个方法: `prepare_prompts` / `prepare_start_tokens` / `prepare_vae_latent` / 
`prepare_vae_latent_cfg` / `prepare_vae_images` / `prepare_vit_images`。

`tensor.to(same_device)` 是 no-op, 多次调用安全, 重复 install 幂等。

In [4]:
from experiments.asym_placement import install_input_transfer_shim

n = install_input_transfer_shim(model, und_device=UND_DEVICE)
assert n == 6, f"expected to shim 6 prepare_* methods, got {n}"
print(f"✅ shim installed on {n} prepare_* methods")

[shim] wrapped 6 prepare_* methods on model → cuda:0
✅ shim installed on 6 prepare_* methods


## 5. InterleaveInferencer + 计时工具

In [5]:
from data.transforms import ImageTransform
from inferencer import InterleaveInferencer, GEN_THINK_SYSTEM_PROMPT

vae_transform = ImageTransform(1024, 512, 16)
vit_transform = ImageTransform(980, 224, 14)
inferencer = InterleaveInferencer(
    model=model, vae_model=vae_model, tokenizer=tokenizer,
    vae_transform=vae_transform, vit_transform=vit_transform,
    new_token_ids=new_token_ids,
)
print(f"inferencer ready")


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def reset_taylorseer_state(m):
    lm = m.language_model.model
    lm.enable_taylorseer = False
    for attr in ("cache_dic", "current"):
        if hasattr(lm, attr):
            delattr(lm, attr)
    for layer in lm.layers:
        layer.enable_taylorseer = False
        for attr in ("cache_dic", "current"):
            if hasattr(layer, attr):
                delattr(layer, attr)
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


class _Elapsed:
    elapsed = None


@contextmanager
def sync_timer():
    torch.cuda.synchronize()
    result = _Elapsed()
    t0 = time.perf_counter()
    yield result
    torch.cuda.synchronize()
    result.elapsed = time.perf_counter() - t0

inferencer ready


## 6. `run_trial` — 测 t_prefill / t_think / t_image

复用 `run_cap_sweep_mp.py:400-470` 结构: 预填 → think (gen_text) → image (gen_image)。
记录每个阶段的耗时, 加上 think tok/s (think 加速的关键指标)。

In [6]:
def run_trial(prompt, cond, seed):
    """Run a single trial. Returns a dict with timings and metadata."""
    reset_taylorseer_state(model)
    set_all_seeds(seed)

    record = dict(prompt=prompt, seed=seed, **cond)
    gen_context = cfg_text_context = cfg_img_context = None
    try:
        gen_context = inferencer.init_gen_context()
        cfg_text_context = deepcopy(gen_context)
        cfg_img_context = deepcopy(gen_context)

        with torch.autocast(device_type="cuda", enabled=True, dtype=torch.bfloat16):
            with sync_timer() as t_prefill:
                gen_context = inferencer.update_context_text(GEN_THINK_SYSTEM_PROMPT, gen_context)
                cfg_img_context = inferencer.update_context_text(GEN_THINK_SYSTEM_PROMPT, cfg_img_context)
                cfg_text_context = deepcopy(gen_context)
                gen_context = inferencer.update_context_text(prompt, gen_context)
                cfg_img_context = inferencer.update_context_text(prompt, cfg_img_context)

            with sync_timer() as t_think:
                gen_text = inferencer.gen_text(
                    gen_context, do_sample=False, temperature=0.3,
                    max_length=cond["max_think_token_n"],
                    min_length=cond.get("min_think_token_n", 0),
                    wait_interjection=cond.get("wait_interjection"),
                )

            gen_context = inferencer.update_context_text(gen_text, gen_context)

            with sync_timer() as t_image:
                inferencer.gen_image(
                    (1024, 1024), gen_context,
                    cfg_text_precontext=cfg_text_context,
                    cfg_img_precontext=cfg_img_context,
                    cfg_text_scale=cond["cfg_text_scale"],
                    cfg_img_scale=cond["cfg_img_scale"],
                    cfg_interval=cond["cfg_interval"],
                    cfg_renorm_min=cond["cfg_renorm_min"],
                    cfg_renorm_type=cond["cfg_renorm_type"],
                    timestep_shift=cond["timestep_shift"],
                    num_timesteps=cond["num_timesteps"],
                    enable_taylorseer=cond["enable_taylorseer"],
                )

        think_token_count = len(tokenizer(gen_text, add_special_tokens=False).input_ids)
        hit_cap = think_token_count >= cond["max_think_token_n"] - 2
        think_closed = gen_text.strip().endswith("</think>")
        tok_per_sec = think_token_count / t_think.elapsed if t_think.elapsed > 0 else None

        record.update(
            t_prefill=t_prefill.elapsed,
            t_think=t_think.elapsed,
            t_image=t_image.elapsed,
            think_token_count=think_token_count,
            think_tok_per_sec=tok_per_sec,
            hit_cap=hit_cap,
            think_closed=think_closed,
            ok=True, error=None,
        )
    except Exception as e:
        record.update(
            t_prefill=None, t_think=None, t_image=None,
            think_token_count=None, think_tok_per_sec=None,
            hit_cap=None, think_closed=None,
            ok=False, error=repr(e),
        )
    finally:
        del gen_context, cfg_text_context, cfg_img_context
        reset_taylorseer_state(model)
        gc.collect()
        torch.cuda.empty_cache()
    return record

## 7. Warmup

先跑 1 次简单 prompt, 触发 kernel 编译, 验证 P0 shim 真的工作。
如果这里 device-mismatch 报错, 看 ERROR 列对照下面的常见原因。

In [7]:
warmup_cond = dict(
    num_timesteps=10, max_think_token_n=64,
    min_think_token_n=64,  # budget forcing
    wait_interjection=" Wait,",
    cfg_text_scale=1.0, cfg_img_scale=1.0,
    cfg_interval=[0.4, 1.0], cfg_renorm_min=0.0, cfg_renorm_type="global",
    timestep_shift=3.0, enable_taylorseer=False,
)
print(f"[warmup] cap=64, num_timesteps=10")
w = run_trial("a photo of a cat", warmup_cond, seed=999)
if not w["ok"]:
    print(f"❌ WARMUP FAILED: {w['error']}")
    print("\n常见原因:")
    print("  - 漏装 P0 shim → 重新跑 Section 4")
    print("  - qwen2_navit.py 补丁未 commit → git log 看 73bc32e")
    print("  - 显存不足 → 调小 num_timesteps 或 cap")
else:
    print(f"✅ warmup ok:")
    print(f"  t_prefill = {w['t_prefill']:.2f}s")
    print(f"  t_think   = {w['t_think']:.2f}s  ({w['think_tok_per_sec']:.1f} tok/s)")
    print(f"  t_image   = {w['t_image']:.2f}s")
    print(f"  think_tokens = {w['think_token_count']}  (hit_cap={w['hit_cap']})")

[warmup] cap=64, num_timesteps=10


100%|██████████| 9/9 [00:10<00:00,  1.15s/it]


✅ warmup ok:
  t_prefill = 3.70s
  t_think   = 2.85s  (22.1 tok/s)
  t_image   = 15.36s
  think_tokens = 63  (hit_cap=True)


## 8. Benchmark sweep

跑一个小网格: 4 prompts × 2 caps × 2 num_timesteps × 1 trial = 16 次。
基线: t_think ≈ 0.055 s/token (18.2 tok/s)。
目标: ≥35 tok/s。

In [8]:
PROMPTS = [
    "a photo of a cute cat sitting on a windowsill",
    "an oil painting of a mountain landscape at sunset",
    "a futuristic cityscape with neon lights at night",
    "a still life of fruits on a wooden table",
]
CAPS = [256, 1000]
NUM_TIMESTEPS = [50, 10]
N_TRIALS = 1  # 想加多次重复改这里
SEED_BASE = 1000

rows = []
t_start = time.perf_counter()
total = len(PROMPTS) * len(CAPS) * len(NUM_TIMESTEPS) * N_TRIALS
i = 0
for pi, prompt in enumerate(PROMPTS):
    for cap in CAPS:
        for nts in NUM_TIMESTEPS:
            for rep in range(N_TRIALS):
                i += 1
                cond = dict(
                    num_timesteps=nts, max_think_token_n=cap,
                    min_think_token_n=cap, wait_interjection=" Wait,",
                    cfg_text_scale=1.0, cfg_img_scale=1.0,
                    cfg_interval=[0.4, 1.0], cfg_renorm_min=0.0, cfg_renorm_type="global",
                    timestep_shift=3.0, enable_taylorseer=False,
                )
                seed = SEED_BASE + pi * 10000 + cap + nts * 10 + rep
                row = run_trial(prompt, cond, seed=seed)
                row["prompt_idx"] = pi
                row["cond_idx"] = cap * 10 + nts
                row["repeat"] = rep
                row["trial_idx"] = i
                rows.append(row)
                elapsed = time.perf_counter() - t_start
                eta = elapsed / i * (total - i) if i < total else 0
                status = "ok" if row["ok"] else f"FAIL: {row['error'][:40]}"
                print(f"[{i:2d}/{total}] cap={cap:4d} ts={nts:2d} prompt#{pi}: {status}  t_think={row.get('t_think')}  tok/s={row.get('think_tok_per_sec')}  elapsed={elapsed:.0f}s ETA={eta:.0f}s")

100%|██████████| 49/49 [00:54<00:00,  1.11s/it]


[ 1/16] cap= 256 ts=50 prompt#0: ok  t_think=9.84774910658598  tok/s=25.894242150164146  elapsed=65s ETA=978s


100%|██████████| 9/9 [00:09<00:00,  1.11s/it]


[ 2/16] cap= 256 ts=10 prompt#0: ok  t_think=9.641920022666454  tok/s=26.44701464029363  elapsed=86s ETA=600s


100%|██████████| 49/49 [00:54<00:00,  1.12s/it]


...

[14/16] cap= 256 ts=10 prompt#3: ok  t_think=9.498240776360035  tok/s=26.847076843394404  elapsed=768s ETA=110s


100%|██████████| 49/49 [00:54<00:00,  1.12s/it]


[15/16] cap=1000 ts=50 prompt#3: ok  t_think=37.71394897252321  tok/s=26.4888728763946  elapsed=862s ETA=57s


100%|██████████| 9/9 [00:10<00:00,  1.12s/it]


[16/16] cap=1000 ts=10 prompt#3: ok  t_think=37.60745192319155  tok/s=26.56388425465067  elapsed=910s ETA=0s


## 9. 结果汇总

In [9]:
df = pd.DataFrame(rows)
df["hardware"] = torch.cuda.get_device_name(0)
df["timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%S")

# 存盘
out_dir = os.path.join(_proj_root, "experiments", "asym_bench_outputs")
os.makedirs(out_dir, exist_ok=True)
out_csv = os.path.join(out_dir, "asym_notebook_trials.csv")
df.to_csv(out_csv, index=False)
print(f"wrote {len(df)} rows → {out_csv}\n")

# 屏幕摘要
ok = df[df["ok"]]
if len(ok) == 0:
    print("❌ no successful trials")
else:
    grp = ok.groupby(["max_think_token_n", "num_timesteps"]).agg(
        t_think_mean=("t_think", "mean"),
        t_think_std=("t_think", "std"),
        tok_per_sec_mean=("think_tok_per_sec", "mean"),
        tok_per_sec_std=("think_tok_per_sec", "std"),
        t_image_mean=("t_image", "mean"),
        t_image_std=("t_image", "std"),
        n=("t_think", "count"),
    )
    print(grp.to_string())
    print()
    overall_tps = ok["think_tok_per_sec"].mean()
    print(f"baseline t_think = 0.055 s/token (18.2 tok/s)  [from compass 报告]")
    print(f"this run mean think tok/s = {overall_tps:.1f}")
    if overall_tps >= 35:
        print(f"✅ ≥35 tok/s 目标达成 (实际 {overall_tps:.1f})")
    else:
        print(f"❌ 低于 35 tok/s 目标 (实际 {overall_tps:.1f})")

wrote 16 rows → /home/wuwenxuan03/bagel/experiments/asym_bench_outputs/asym_notebook_trials.csv

                                 t_think_mean  t_think_std  tok_per_sec_mean  tok_per_sec_std  t_image_mean  t_image_std  n
max_think_token_n num_timesteps                                                                                            
256               10                 9.553365     0.079411         26.693547         0.221715     10.339708     0.008084  4
                  50                 9.667534     0.121105         26.380012         0.326563     54.862111     0.021178  4
1000              10                37.544933     0.087707         26.608227         0.062256     10.437354     0.009051  4
                  50                37.629953     0.070375         26.548070         0.049657     55.345259     0.030185  4

baseline t_think = 0.055 s/token (18.2 tok/s)  [from compass 报告]
this run mean think tok/s = 26.6
❌ 低于 35 tok/s 目标 (实际 26.6)


## 10. `nvidia-smi` 快照

期望: think 阶段 GPU 1 利用率 ≈ 0 (因为 gen 专家只在 image 生成阶段被用到, 
而 think 阶段只走 und 路径, 全部 cuda:0)。如果 GPU 1 在 think 时也在忙, 
说明跨卡通信没消除。

In [10]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Tue Jul  7 11:49:12 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.54.03              Driver Version: 535.54.03    CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4090        On  | 00000000:01:00.0 Off |                  Off |
| 30%   35C    P5              76W / 450W |  16235MiB / 24564MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## 11. 总结与下一步

**已完成**:
- asym placement 加载器 (8.165B/15.37 GiB on cuda:0, 6.526B/12.15 GiB on cuda:1)
- 静态 verify_placement (零错放)
- P0 input transfer shim (6 个 prepare_* 方法)
- qwen2_navit.py 4 处 gen 分支 device 搬运补丁

**需要在本机运行观察**:
- warmup 是否通过 (Section 7) — 验证 P0 shim 真的工作
- 16 次 trial 的 `think_tok_per_sec` 中位数 — 是否达到 ≥35 tok/s 目标
- `t_image` 回归幅度 — 已知代价, 阶段 0 接受
- `nvidia-smi` 在 think 阶段 GPU 1 利用率 — 应 ≈ 0

**下一步 (阶段 0 完成后)**:
1. 与 pipeline 模式 (13/15 accelerate 加载) 跑同一组 prompts, 对比 CSV
2. 数字写入 `EXPERIMENT_SUMMARY.md` / `DEBUG_NOTES.md`
3. 阶段 1: 解决 t_image 回归 (备选方案: 把 hidden states 主体放 GPU 1, 
   只把少量文本 token 切片搬 GPU 0, KV cache 一次性搬 GPU 1)